In [28]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Tải dữ liệu
df = pd.read_csv('Agri_Data_Cleaned.csv')

# 2. Tách các cột số (Trừ cột mục tiêu 'Yield' ra nếu bạn muốn giữ nguyên target)
# Lưu ý: Với hồi quy, thường ta chỉ chuẩn hóa Input (X), không nhất thiết chuẩn hóa Output (y)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
if 'Yield' in numeric_cols:
    numeric_cols.remove('Yield')

# 3. Khởi tạo bộ chuẩn hóa Z-score
scaler = StandardScaler()

# 4. Thực hiện chuẩn hóa
# Tạo dataframe mới để không ảnh hưởng dữ liệu gốc
df_scaled = df.copy()
df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# 5. Kiểm tra kết quả
# Ví dụ: Production cũ là 200 (nhỏ) -> Z-score là -0.22
#        Production cũ là 268754 (lớn) -> Z-score là 3.67
#        -> Thứ tự vẫn được bảo toàn!
print(df_scaled[['Production', 'Avg Temp', 'NDVI_Season_Mean']].head())

# Lưu file kết quả
df_scaled.to_csv('Agri_Data_Normalized.csv', index=False)

   Production  Avg Temp  NDVI_Season_Mean
0   -0.228667 -1.324513          1.301556
1   -0.217816 -0.738050          1.301556
2    3.677861  0.434877          1.301556
3   -0.199574 -0.738050          1.301556
4   -0.203749  1.607804         -1.003848


# Xóa cột AP Ratio, Production , Avg Temp,Avg Humidity,Rain_Temp_Ratio, CN_Ratio, sm_surface

In [29]:
import pandas as pd

# 1. Đọc file dữ liệu
file_path = 'Agri_Data_Normalized.csv'
df = pd.read_csv(file_path)

# 2. Kiểm tra và xóa các cột: 'AP Ratio', 'Production', 'Avg Temp', 'Avg Humidity'
# Danh sách các cột muốn xóa
cols_to_drop = ['AP Ratio', 'Production','Avg Temp','Avg Humidity','Is_Extreme_Heat','is_extreme_Heat_Stress_Days']

# Chỉ xóa những cột thực sự tồn tại trong file để tránh lỗi nếu cột không có
existing_cols = [col for col in cols_to_drop if col in df.columns]

if existing_cols:
    df = df.drop(columns=existing_cols)
    print(f"Đã xóa các cột: {existing_cols}")
else:
    print("Không tìm thấy các cột cần xóa trong file.")

# 3. Lưu kết quả ra file mới (để giữ nguyên file gốc)
output_file = 'Agri_Data_Cleaned_Processed.csv'
df.to_csv(output_file, index=False)

print(f"File đã được lưu tại: {output_file}")

# Hiển thị 5 dòng đầu để kiểm tra
print(df.head())

Đã xóa các cột: ['AP Ratio', 'Production', 'Avg Temp', 'Avg Humidity', 'Is_Extreme_Heat', 'is_extreme_Heat_Stress_Days']
File đã được lưu tại: Agri_Data_Cleaned_Processed.csv
       Area  District    Season     Crop Name Transplant        Growth  \
0 -0.195962  Bagerhat      Rabi         Wheat   December  Jan To March   
1 -0.192295  Bagerhat      Rabi       Maize 2   December  Jan To March   
2  3.090006  Bagerhat      Rabi          Boro   November  Dec To March   
3 -0.187633  Bagerhat      Rabi  Sweet Potato   November  Dec To March   
4 -0.193155  Bagerhat  Kharif 1         Mango      April  April To May   

         Harvest  Max Temp  Min Temp  Max Relative Humidity  ...  \
0          April -1.843403 -0.287335              -1.288481  ...   
1          April -0.656067 -0.564733              -1.288481  ...   
2          April  1.548984 -0.934598              -0.298672  ...   
3          April -1.164925 -0.009937               0.691137  ...   
4  April To June  0.870507  1.839384    

In [30]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

def super_optimize_features(file_path, target_col='Yield'):
    print(f"--- BẮT ĐẦU TỐI ƯU HÓA SÂU: {file_path} ---")
    
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{file_path}'")
        return

    # 1. Xác định các cột vẫn còn yếu (tương quan < 0.05)
    # Chúng ta lọc ra các cột số để kiểm tra
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if target_col in numeric_cols: numeric_cols.remove(target_col)
    
    # Tính tương quan hiện tại
    current_corrs = df[numeric_cols].corrwith(df[target_col]).abs()
    
    # Lấy danh sách các cột yếu (dưới 0.05)
    weak_cols = current_corrs[current_corrs < 0.05].index.tolist()
    
    print(f"Phát hiện {len(weak_cols)} cột yếu cần xử lý lại: {weak_cols}")
    
    # Danh sách các "chìa khóa" để thử ghép (Grouping Keys)
    # Đây là các biến phân loại có thể gây ra hiệu ứng Simpson
    potential_keys = ['Crop Name', 'Season', 'District', 'Dominant_Soil_Texture', 'Water_Availability_Cat']
    # Chỉ giữ lại các key thực sự có trong dataframe
    grouping_keys = [k for k in potential_keys if k in df.columns]
    
    results_log = []
    cols_to_drop = []

    # 2. Vòng lặp "Thử sai" (Trial & Error) cho từng cột yếu
    for col in weak_cols:
        best_corr = current_corrs[col]
        best_key = None
        best_new_col_name = None
        best_slope_map = None
        
        # Thử ghép cột yếu này với từng key (Season, District, Soil, v.v.)
        for key in grouping_keys:
            # Bỏ qua nếu cột đó chính là key (hoặc tương tự)
            if col == key: continue
            
            # Tính Slope cho từng nhóm của key này
            slopes = {}
            
            groups = df[key].unique()
            for group in groups:
                subset = df[df[key] == group]
                # Chỉ tính nếu nhóm có đủ dữ liệu (>5 mẫu)
                if len(subset) > 5:
                    # Nếu cột này là hằng số trong nhóm (độ lệch chuẩn = 0) -> Slope = 0
                    if subset[col].std() == 0: 
                        slopes[group] = 0
                    else:
                        model = LinearRegression()
                        model.fit(subset[[col]], subset[target_col])
                        slopes[group] = model.coef_[0]
                else:
                    slopes[group] = 0
            
            # Tạo cột giả định để kiểm tra hiệu quả
            # Công thức: Giá trị gốc * Slope của nhóm
            temp_col = df[col] * df[key].map(slopes).fillna(0)
            temp_corr = temp_col.corr(df[target_col])
            
            # Ghi nhận nếu kết quả tốt hơn kỷ lục hiện tại
            if abs(temp_corr) > best_corr:
                best_corr = abs(temp_corr)
                best_key = key
                best_new_col_name = f"{col}_x_{key}_Optimized"
                best_slope_map = slopes

        # 3. Áp dụng thay đổi tốt nhất (nếu có cải thiện đáng kể > 0.01)
        if best_key and (best_corr > current_corrs[col] + 0.01):
            print(f"  [FIX] {col}: Tương quan {current_corrs[col]:.3f} -> {best_corr:.3f} (Ghép với '{best_key}')")
            
            # Tạo cột mới chính thức trong DataFrame
            df[best_new_col_name] = df[col] * df[best_key].map(best_slope_map).fillna(0)
            
            results_log.append({
                'Feature': col,
                'Optimized_By': best_key,
                'New_Corr': best_corr
            })
            
            # Đánh dấu xóa cột cũ để tránh trùng lặp thông tin
            if col not in cols_to_drop:
                cols_to_drop.append(col)
        else:
            print(f"  [SKIP] {col}: Không tìm thấy cách ghép nào hiệu quả (Max {best_corr:.3f}). Có thể do dữ liệu nhiễu hoàn toàn.")

    # 4. Xóa cột cũ và Lưu file
    if cols_to_drop:
        print(f"\nĐang xóa {len(cols_to_drop)} cột gốc yếu...")
        df.drop(columns=cols_to_drop, inplace=True)
    
    # Đặt tên file đầu ra
    output_file = 'Agri_Data_Final_Optimized.csv'
    df.to_csv(output_file, index=False)
    print(f"\n--- HOÀN TẤT! Đã lưu file tối ưu tại: {output_file} ---")
    print(f"Tổng số cột hiện tại: {df.shape[1]}")

# --- CHẠY HÀM ---
# Lưu ý: Đảm bảo file 'Agri_Data_Cleaned_Processed.csv' nằm cùng thư mục
super_optimize_features('Agri_Data_Cleaned_Processed.csv')

--- BẮT ĐẦU TỐI ƯU HÓA SÂU: Agri_Data_Cleaned_Processed.csv ---
Phát hiện 21 cột yếu cần xử lý lại: ['Area', 'Max Relative Humidity', 'EVI', 'Soil_Moisture_mm', 'NDVI_Season_Std', 'NDVI_Season_Range', 'pH', 'Organic_Carbon', 'Nitrogen', 'Clay', 'Sand', 'Silt', 'Bulk_Density', 'CN_Ratio', 'sm_surface', 'sm_rootzone', 'Rootzone_Surface_Diff', 'Wind_Mean', 'Wind_Max', 'Rain_Temp_Ratio', 'is_extreme_Wind_Max']
  [FIX] Area: Tương quan 0.034 -> 0.170 (Ghép với 'Crop Name')
  [FIX] Max Relative Humidity: Tương quan 0.016 -> 0.078 (Ghép với 'District')
  [FIX] EVI: Tương quan 0.045 -> 0.311 (Ghép với 'Crop Name')
  [FIX] Soil_Moisture_mm: Tương quan 0.017 -> 0.111 (Ghép với 'Crop Name')
  [FIX] NDVI_Season_Std: Tương quan 0.033 -> 0.133 (Ghép với 'Crop Name')
  [FIX] NDVI_Season_Range: Tương quan 0.028 -> 0.116 (Ghép với 'Crop Name')
  [FIX] pH: Tương quan 0.027 -> 0.085 (Ghép với 'Crop Name')
  [FIX] Organic_Carbon: Tương quan 0.011 -> 0.045 (Ghép với 'Crop Name')
  [FIX] Nitrogen: Tương qua